In [1]:
import os

os.makedirs("../csv", exist_ok=True)
os.makedirs("../svg", exist_ok=True)

In [2]:
import pandas as pd
import numpy as np
from scipy import stats
import matplotlib as mpl
import matplotlib.pyplot as plt
import seaborn as sns
import scienceplots

In [3]:
plt.rcParams.update({
    'font.family': 'TeX Gyre Termes',
    'font.size': 11,
    'axes.labelsize': 11,
    'axes.titlesize': 12,
    'figure.dpi': 300,
    'savefig.dpi': 300,
})

In [4]:
TEXTWIDTH_PT = 426.79137
TEXTHEIGHT_PT = 702.78308
PT_PER_INCH = 72.27

FIG_WIDTH = TEXTWIDTH_PT / PT_PER_INCH
FIG_HEIGHT_1 = TEXTHEIGHT_PT / PT_PER_INCH
FIG_HEIGHT_2 = FIG_HEIGHT_1 / 2
FIG_HEIGHT_3 = FIG_HEIGHT_1 / 3
FIG_HEIGHT_4 = FIG_HEIGHT_1 / 4
FIG_HEIGHT_5 = FIG_HEIGHT_1 / 5
FIG_HEIGHT_6 = FIG_HEIGHT_1 / 6
FIG_HEIGHT_7 = FIG_HEIGHT_1 / 7
FIG_HEIGHT_8 = FIG_HEIGHT_1 / 8

In [5]:
backendy_pl = {
    'scalar': 'skalarny',
    'sse2': 'SSE2',
    'avx2': 'AVX2',
    'avx512': 'AVX-512',
    'neon': 'NEON',
}

In [6]:
wersje_pl = {
    'Speck32_64':   '32/64',
    'Speck48_72':   '48/72',
    'Speck48_96':   '48/96',
    'Speck64_96':   '64/96',
    'Speck64_128':  '64/128',
    'Speck96_96':   '96/96',
    'Speck96_144':  '96/144',
    'Speck128_128': '128/128',
    'Speck128_192': '128/192',
    'Speck128_256': '128/256',
}

In [7]:
backend_order = ['scalar', 'sse2', 'avx2', 'avx512']
version_order = ['32_64', '48_72', '48_96', '64_96', '64_128', '96_96', '96_144', '128_128', '128_192', '128_256']

In [8]:
system_x86 = pd.read_csv('../data/system_x86.csv')
system_aarch64 = pd.read_csv('../data/system_aarch64.csv')

In [9]:
system_x86['time_per_key_ns'] = system_x86['duration_ns'] / system_x86['throughput_num']
system_x86['keys_per_sec'] = 1e9 / system_x86['time_per_key_ns']

print(system_x86)

      bits_measured benchmark backend architecture cipher_mode  \
0                12    system  Avx512       x86_64         Ecb   
1                13    system  Avx512       x86_64         Ecb   
2                14    system  Avx512       x86_64         Ecb   
3                15    system  Avx512       x86_64         Ecb   
4                16    system  Avx512       x86_64         Ecb   
...             ...       ...     ...          ...         ...   
1115             22    system  Scalar       x86_64         Cbc   
1116             23    system  Scalar       x86_64         Cbc   
1117             24    system  Scalar       x86_64         Cbc   
1118             28    system  Scalar       x86_64         Cbc   
1119             32    system  Scalar       x86_64         Cbc   

             function       version  suffix  throughput_num unit  duration_ns  \
0     EncryptInflight    Speck32_64       1            4096   ns       714697   
1     EncryptInflight    Speck32_64       1  

In [10]:
system_aarch64

,bits_measured,benchmark,backend,architecture,cipher_mode,function,version,suffix,throughput_num,unit,duration_ns
0,11,system,Neon,aarch64,Ecb,EncryptInflight,Speck32_64,1,2048,ns,583416
1,12,system,Neon,aarch64,Ecb,EncryptInflight,Speck32_64,1,4096,ns,244125
2,13,system,Neon,aarch64,Ecb,EncryptInflight,Speck32_64,1,8192,ns,231375
3,14,system,Neon,aarch64,Ecb,EncryptInflight,Speck32_64,1,16384,ns,322917
4,15,system,Neon,aarch64,Ecb,EncryptInflight,Speck32_64,1,32768,ns,417041
...,...,...,...,...,...,...,...,...,...,...,...
555,21,system,Scalar,aarch64,Cbc,EncryptInflight,Speck128_256,2,2097152,ns,10501041
556,22,system,Scalar,aarch64,Cbc,EncryptInflight,Speck128_256,2,4194304,ns,20446291
557,23,system,Scalar,aarch64,Cbc,EncryptInflight,Speck128_256,2,8388608,ns,39452542
558,27,system,Scalar,aarch64,Cbc,EncryptInflight,Speck128_256,2,134217728,ns,612333292


In [29]:
import pandas as pd
import numpy as np
from scipy import stats

system_x86 = pd.read_csv('../data/system_x86.csv')

GROUP_COLS = ['backend', 'function', 'version', 'suffix', 'cipher_mode']

def key_bits_from_version(v):
    return int(str(v).split('_')[1])

def fit_group(g):
    g = g.sort_values('throughput_num')
    n = g['throughput_num'].to_numpy(dtype=float)
    t = g['duration_ns'].to_numpy(dtype=float)

    if len(g) < 3:
        return None

    kb = key_bits_from_version(g['version'].iloc[0])
    N_full = 2.0 ** kb

    slope, intercept, lo, hi = stats.theilslopes(t, n, 0.90)
    t_key = slope
    thr = 1e9 / t_key if t_key > 0 else np.nan

    tau, p_tau = stats.kendalltau(n, t)

    pred_total_ns = intercept + slope * N_full
    pred_total_s = pred_total_ns * 1e-9
    span = n.max() / n.min()

    return {
        'key_bits': kb,
        'overhead_ns': intercept,
        'ns_per_key': t_key,
        'ns_per_key_lo': 1e9 / hi if hi > 0 else np.nan,
        'ns_per_key_hi': 1e9 / lo if lo > 0 else np.nan,
        'throughput_keys_per_s': thr,
        'kendall_tau': tau,
        'p_value_kendall': p_tau,
        'full_keyspace': N_full,
        'pred_total_seconds': pred_total_s,
        'pred_total_years': pred_total_s / (3600 * 24 * 365.25),
        'reliable': (t_key > 0) and (span >= 8) and (p_tau < 0.05),
    }

rows = []
for keys, g in system_x86.groupby(GROUP_COLS):
    res = fit_group(g)
    if res is not None:
        rows.append(dict(zip(GROUP_COLS, keys)) | res)

predictions = pd.DataFrame(rows).sort_values(GROUP_COLS).reset_index(drop=True)
predictions

,backend,function,version,suffix,cipher_mode,key_bits,overhead_ns,ns_per_key,ns_per_key_lo,ns_per_key_hi,throughput_keys_per_s,kendall_tau,p_value_kendall,full_keyspace,pred_total_seconds,pred_total_years,reliable
0,Avx2,EncryptInflight,Speck128_128,1,Cbc,128,157915.486804,1.229111,7.721405e+08,8.284901e+08,8.135961e+08,1.000000,0.000397,3.402824e+38,4.182448e+29,1.325338e+22,True
1,Avx2,EncryptInflight,Speck128_128,1,Ecb,128,167145.100000,1.105527,8.596119e+08,6.581241e+09,9.045462e+08,0.619048,0.069048,3.402824e+38,3.761913e+29,1.192078e+22,False
2,Avx2,EncryptInflight,Speck128_128,2,Cbc,128,44066.333333,1.111407,8.681073e+08,9.139313e+08,8.997607e+08,1.000000,0.000397,3.402824e+38,3.781921e+29,1.198418e+22,True
3,Avx2,EncryptInflight,Speck128_128,2,Ecb,128,836440.764706,1.147483,8.521101e+08,8.904068e+08,8.714723e+08,1.000000,0.000397,3.402824e+38,3.904684e+29,1.237320e+22,True
4,Avx2,EncryptInflight,Speck128_192,1,Cbc,192,159205.347826,1.121571,8.179458e+08,9.297469e+08,8.916062e+08,0.714286,0.030159,6.277102e+57,7.040218e+48,2.230910e+41,True
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
155,Sse2,EncryptInflight,Speck96_144,2,Ecb,144,-121817.000000,2.727864,3.082128e+08,3.728844e+08,3.665872e+08,1.000000,0.000397,2.230075e+43,6.083341e+34,1.927694e+27,True
156,Sse2,EncryptInflight,Speck96_96,1,Cbc,96,150023.258065,2.755638,3.620840e+08,4.114102e+08,3.628924e+08,1.000000,0.000397,7.922816e+28,2.183241e+20,6.918274e+12,True
157,Sse2,EncryptInflight,Speck96_96,1,Ecb,96,149166.937255,2.799807,3.499646e+08,4.809770e+08,3.571675e+08,0.904762,0.002778,7.922816e+28,2.218235e+20,7.029164e+12,True
158,Sse2,EncryptInflight,Speck96_96,2,Cbc,96,133839.000000,2.608451,3.773327e+08,3.843360e+08,3.833693e+08,1.000000,0.000397,7.922816e+28,2.066628e+20,6.548748e+12,True


In [30]:
import os
import numpy as np

INDEX_COLS = ["backend", "version", "cipher_mode"]
SUFFIXES   = sorted(predictions["suffix"].unique())

def fmt_pow2(x):
    if pd.isna(x) or x <= 0:
        return "--"
    return rf"\(2^{{{np.log2(x):.2f}}}\)"

def fmt_sci(x):
    if pd.isna(x):
        return "--"
    m, e = f"{x:.3e}".split("e")
    return rf"\({float(m):.2f}\times10^{{{int(e)}}}\)"

def fmt_pow2_pct(x, lo, hi):
    """Throughput jako 2^a z asymetryczną niepewnością procentową: 2^a (^{+u%}_{-d%})."""
    if pd.isna(x) or x <= 0:
        return "--"
    a = np.log2(x)
    if pd.isna(lo) or pd.isna(hi) or lo <= 0 or hi <= 0:
        return rf"\(2^{{{a:.2f}}}\)"
    up = (hi - x) / x * 100      # lo/hi to throughput dolny/górny (keys/s)
    dn = (x - lo) / x * 100
    return rf"\(2^{{{a:.2f}}}\,(^{{+{up:.1f}\%}}_{{-{dn:.1f}\%}})\)"

# --- formatowanie PRZED pivotem ---
predictions = predictions.copy()
predictions["throughput_fmt"] = [
    fmt_pow2_pct(x, lo, hi)
    for x, lo, hi in zip(
        predictions["throughput_keys_per_s"],
        predictions["ns_per_key_lo"],   # = throughput dolny (1e9/hi_slope)
        predictions["ns_per_key_hi"],   # = throughput górny (1e9/lo_slope)
    )
]

wide = predictions.set_index(INDEX_COLS + ["suffix"])

cols = {}
for suf in SUFFIXES:
    sub = wide.xs(suf, level="suffix")
    cols[f"{suf}_throughput"] = sub["throughput_fmt"]

latex_df = pd.DataFrame(cols).reset_index()

# --- klucze sortujące z SUROWYCH wartości (przed tłumaczeniem) ---
latex_df["_bk"] = pd.Categorical(
    latex_df["backend"].astype(str).str.lower(),
    categories=backend_order, ordered=True)
latex_df["_ver"] = pd.Categorical(
    latex_df["version"].astype(str).str.replace("Speck", "", regex=False),
    categories=version_order, ordered=True)

latex_df = latex_df.sort_values(["_bk", "_ver"]).drop(columns=["_bk", "_ver"])

# --- dopiero teraz tłumaczenia na tekst ---
latex_df["backend"] = latex_df["backend"].astype(str).str.lower().map(backendy_pl).fillna(latex_df["backend"])
latex_df["version"] = latex_df["version"].map(wersje_pl).fillna(latex_df["version"])

latex_df = latex_df.reset_index(drop=True)

os.makedirs("../csv", exist_ok=True)
latex_df.to_csv("../csv/prediction_results.csv", index=False)
latex_df

,backend,version,cipher_mode,1_throughput,2_throughput
0,skalarny,32/64,Cbc,"\(2^{29.35}\,(^{+4.0\%}_{-5.6\%})\)","\(2^{29.31}\,(^{+4.4\%}_{-2.6\%})\)"
1,skalarny,32/64,Ecb,"\(2^{29.36}\,(^{+21.1\%}_{-0.5\%})\)","\(2^{29.30}\,(^{+5.5\%}_{-18.0\%})\)"
2,skalarny,48/72,Cbc,"\(2^{28.36}\,(^{+7.2\%}_{-16.4\%})\)","\(2^{28.35}\,(^{+0.1\%}_{-5.6\%})\)"
3,skalarny,48/72,Ecb,"\(2^{28.37}\,(^{+16.6\%}_{-0.9\%})\)","\(2^{28.35}\,(^{+0.7\%}_{-11.8\%})\)"
4,skalarny,48/96,Cbc,"\(2^{28.33}\,(^{+1.3\%}_{-2.5\%})\)","\(2^{28.31}\,(^{+1.9\%}_{-9.7\%})\)"
...,...,...,...,...,...
75,AVX-512,128/128,Ecb,\(2^{30.21}\),"\(2^{30.17}\,(^{+0.2\%}_{-3.4\%})\)"
76,AVX-512,128/192,Cbc,"\(2^{30.19}\,(^{+48.1\%}_{-5.9\%})\)","\(2^{30.14}\,(^{+3.2\%}_{-3.4\%})\)"
77,AVX-512,128/192,Ecb,\(2^{30.14}\),"\(2^{30.10}\,(^{+7.0\%}_{-2.4\%})\)"
78,AVX-512,128/256,Cbc,"\(2^{30.13}\,(^{+13.6\%}_{-24.3\%})\)","\(2^{30.13}\,(^{+2.2\%}_{-12.6\%})\)"


In [13]:
import numpy as np

# różnica log2 throughputu: dodatnia => suffix 2 szybszy
piv = predictions.pivot_table(
    index=["backend", "version"],
    columns="suffix",
    values="throughput_keys_per_s",
    observed=True,
)

cmp = piv.reset_index()
cmp.columns = ["backend", "version", "thr_s1", "thr_s2"]

cmp["delta_log2"]        = np.log2(cmp["thr_s2"] / cmp["thr_s1"])   # +1.0 = 2x szybciej
cmp["speedup_s2_over_s1"] = cmp["thr_s2"] / cmp["thr_s1"]           # krotność
cmp["winner"] = np.where(cmp["delta_log2"] > 0, "suffix 2",
                np.where(cmp["delta_log2"] < 0, "suffix 1", "remis"))

suffix_analysis = cmp.sort_values("delta_log2", ascending=False).reset_index(drop=True)
suffix_analysis

,backend,version,thr_s1,thr_s2,delta_log2,speedup_s2_over_s1,winner
0,Avx512,Speck32_64,1.560660e+09,6.871420e+09,2.138452,4.402893,suffix 2
1,Avx2,Speck32_64,1.693852e+09,3.806050e+09,1.167987,2.246979,suffix 2
2,Avx512,Speck64_128,1.673572e+09,2.881094e+09,0.783686,1.721524,suffix 2
3,Avx2,Speck64_128,1.284877e+09,2.115987e+09,0.719701,1.646840,suffix 2
4,Avx512,Speck64_96,1.801747e+09,2.956479e+09,0.714483,1.640896,suffix 2
5,Avx2,Speck96_96,4.888696e+08,7.189237e+08,0.556389,1.470584,suffix 2
6,Sse2,Speck32_64,1.836391e+09,2.430439e+09,0.404344,1.323487,suffix 2
7,Avx2,Speck128_256,6.818152e+08,8.925307e+08,0.388521,1.309051,suffix 2
8,Avx2,Speck96_144,5.654153e+08,7.077038e+08,0.323835,1.251653,suffix 2
9,Avx2,Speck128_192,7.246044e+08,9.008530e+08,0.314098,1.243234,suffix 2


In [14]:
wins = suffix_analysis["winner"].value_counts()

summary = pd.DataFrame({
    "metryka": [
        "wierszy ogółem", "wygrane suffix 2", "wygrane suffix 1", "remisy",
        "śr. przewaga suffix 2 [×]", "mediana przewagi [×]",
        "maks. przewaga suffix 2 [×]", "maks. przewaga suffix 1 [×]",
    ],
    "wartość": [
        len(suffix_analysis),
        int(wins.get("suffix 2", 0)),
        int(wins.get("suffix 1", 0)),
        int(wins.get("remis", 0)),
        2.0 ** suffix_analysis["delta_log2"].mean(),
        2.0 ** suffix_analysis["delta_log2"].median(),
        suffix_analysis["speedup_s2_over_s1"].max(),
        1.0 / suffix_analysis["speedup_s2_over_s1"].min(),
    ],
})

per_backend = (suffix_analysis
    .groupby("backend", observed=True)["delta_log2"]
    .agg(["mean", "median", "min", "max"])
    .assign(speedup_mean=lambda d: 2.0 ** d["mean"]))

per_backend

,mean,median,min,max,speedup_mean
backend,,,,,
Avx2,0.373159,0.318967,0.014939,1.167987,1.295185
Avx512,0.379945,0.076982,-0.049233,2.138452,1.301292
Scalar,-0.015129,-0.026098,-0.078493,0.152985,0.989568
Sse2,0.011302,0.011052,-0.248764,0.404344,1.007865


In [27]:
import pandas as pd
import numpy as np
from scipy import stats

compare_x86_avx2 = pd.read_csv('../data/compare_x86_Avx2.csv')
compare_x86_avx512 = pd.read_csv('../data/compare_x86_Avx512.csv')

compare_x86 = pd.concat([compare_x86_avx2, compare_x86_avx512], ignore_index=True)

GROUP_COLS = ['backend', 'function', 'version', 'suffix', 'cipher_mode']

def key_bits_from_version(v):
    return int(str(v).split('_')[1])

def fit_group(g):
    g = g.sort_values('throughput_num')
    n = g['throughput_num'].to_numpy(dtype=float)
    t = g['duration_ns'].to_numpy(dtype=float)

    if len(g) < 3:
        return None

    kb = key_bits_from_version(g['version'].iloc[0])
    N_full = 2.0 ** kb

    slope, intercept, lo, hi = stats.theilslopes(t, n)
    t_key = slope
    thr = 1e9 / t_key if t_key > 0 else np.nan

    tau, p_tau = stats.kendalltau(n, t)

    pred_total_ns = intercept + slope * N_full
    pred_total_s = pred_total_ns * 1e-9
    span = n.max() / n.min()

    return {
        'key_bits': kb,
        'overhead_ns': intercept,
        'ns_per_key': t_key,
        'ns_per_key_lo': 1e9 / hi if hi > 0 else np.nan,
        'ns_per_key_hi': 1e9 / lo if lo > 0 else np.nan,
        'throughput_keys_per_s': thr,
        'kendall_tau': tau,
        'p_value_kendall': p_tau,
        'full_keyspace': N_full,
        'pred_total_seconds': pred_total_s,
        'pred_total_years': pred_total_s / (3600 * 24 * 365.25),
        'reliable': (t_key > 0) and (span >= 8) and (p_tau < 0.05),
    }

rows = []
for keys, g in compare_x86.groupby(GROUP_COLS):
    res = fit_group(g)
    if res is not None:
        rows.append(dict(zip(GROUP_COLS, keys)) | res)

predictions_compare = pd.DataFrame(rows).sort_values(GROUP_COLS).reset_index(drop=True)
predictions_compare

,backend,function,version,suffix,cipher_mode,key_bits,overhead_ns,ns_per_key,ns_per_key_lo,ns_per_key_hi,throughput_keys_per_s,kendall_tau,p_value_kendall,full_keyspace,pred_total_seconds,pred_total_years,reliable
0,Avx2,EncryptInflight,Speck128_128,2,Ecb,128,-3.187554e+06,0.963483,9.816261e+08,1.082048e+09,1.037901e+09,1.000000,0.000006,3.402824e+38,3.278564e+29,1.038914e+22,True
1,Avx2,EncryptInflight,Speck32_64,2,Ecb,64,-5.847698e+05,0.223375,4.216687e+09,4.774413e+09,4.476769e+09,0.944444,0.000050,1.844674e+19,4.120549e+09,1.305723e+02,True
2,Avx2,EncryptInflight,Speck48_72,2,Ecb,72,2.857780e+06,0.521789,1.828029e+09,1.968646e+09,1.916482e+09,1.000000,0.000006,4.722366e+21,2.464081e+12,7.808201e+04,True
3,Avx2,EncryptInflight,Speck64_96,2,Ecb,96,1.066906e+05,0.394815,2.381150e+09,2.579733e+09,2.532833e+09,0.944444,0.000050,7.922816e+28,3.128045e+19,9.912177e+11,True
4,Avx512,EncryptInflight,Speck128_128,2,Ecb,128,3.818635e+05,0.775369,1.200015e+09,1.309446e+09,1.289708e+09,1.000000,0.000006,3.402824e+38,2.638445e+29,8.360728e+21,True
5,Avx512,EncryptInflight,Speck32_64,2,Ecb,64,1.614433e+06,0.131356,7.466754e+09,7.704973e+09,7.612917e+09,0.888889,0.000243,1.844674e+19,2.423085e+09,7.678293e+01,True
6,Avx512,EncryptInflight,Speck48_72,2,Ecb,72,-5.165384e+05,0.479364,1.985123e+09,2.265684e+09,2.086097e+09,0.944444,0.000050,4.722366e+21,2.263733e+12,7.173336e+04,True
7,Avx512,EncryptInflight,Speck64_96,2,Ecb,96,-5.879534e+05,0.293265,3.378305e+09,3.472203e+09,3.409885e+09,0.944444,0.000050,7.922816e+28,2.323485e+19,7.362679e+11,True


In [28]:
import os
import numpy as np

INDEX_COLS = ["backend", "version"]

def fmt_pow2_pct(x, lo, hi):
    if pd.isna(x) or x <= 0:
        return "--"
    a = np.log2(x)
    if pd.isna(lo) or pd.isna(hi) or lo <= 0 or hi <= 0:
        return rf"\(2^{{{a:.2f}}}\)"
    up = (hi - x) / x * 100
    dn = (x - lo) / x * 100
    return rf"\(2^{{{a:.2f}}}\,(^{{+{up:.1f}\%}}_{{-{dn:.1f}\%}})\)"

sub = predictions_compare.set_index(INDEX_COLS)

latex_df = pd.DataFrame({
    "throughput": [
        fmt_pow2_pct(x, lo, hi)
        for x, lo, hi in zip(
            sub["throughput_keys_per_s"],
            sub["ns_per_key_lo"],
            sub["ns_per_key_hi"],
        )
    ],
}, index=sub.index).reset_index()

# --- klucze sortujące z SUROWYCH wartości (przed tłumaczeniem) ---
latex_df["_bk"] = pd.Categorical(
    latex_df["backend"].astype(str).str.lower(),
    categories=backend_order, ordered=True)
latex_df["_ver"] = pd.Categorical(
    latex_df["version"].astype(str).str.replace("Speck", "", regex=False),
    categories=version_order, ordered=True)

latex_df = latex_df.sort_values(["_bk", "_ver"]).drop(columns=["_bk", "_ver"])

# --- dopiero teraz tłumaczenia na tekst ---
latex_df["backend"] = latex_df["backend"].astype(str).str.lower().map(backendy_pl).fillna(latex_df["backend"])
latex_df["version"] = latex_df["version"].map(wersje_pl).fillna(latex_df["version"])

latex_df = latex_df.reset_index(drop=True)

os.makedirs("../csv", exist_ok=True)
latex_df.to_csv("../csv/prediction_results.csv", index=False)
latex_df

,backend,version,throughput
0,AVX2,32/64,"\(2^{32.06}\,(^{+6.6\%}_{-5.8\%})\)"
1,AVX2,48/72,"\(2^{30.84}\,(^{+2.7\%}_{-4.6\%})\)"
2,AVX2,64/96,"\(2^{31.24}\,(^{+1.9\%}_{-6.0\%})\)"
3,AVX2,128/128,"\(2^{29.95}\,(^{+4.3\%}_{-5.4\%})\)"
4,AVX-512,32/64,"\(2^{32.83}\,(^{+1.2\%}_{-1.9\%})\)"
5,AVX-512,48/72,"\(2^{30.96}\,(^{+8.6\%}_{-4.8\%})\)"
6,AVX-512,64/96,"\(2^{31.67}\,(^{+1.8\%}_{-0.9\%})\)"
7,AVX-512,128/128,"\(2^{30.26}\,(^{+1.5\%}_{-7.0\%})\)"
